# Part 11: HMM Model Comparison

This notebook runs the reproducible Part 11 Python runner in Colab. Part 11 audits the frozen Part 1 HMM and KMeans outputs; it does not re-estimate PCA, HMM, or KMeans models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part11_hmm_model_comparison' / 'run_part11_hmm_model_comparison.py').exists(), 'Part 11 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part11_hmm_model_comparison/requirements-part11.txt

## Path configuration

Part 11 requires completed Part 1 and Part 2 runs. The helper first tries the canonical Drive path, then the downloaded `*_outputs/...` folder structure.

In [ ]:
from pathlib import Path

def choose_existing(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART1_RUN_DIR = choose_existing(
    'outputs/part1_btc_macro_state/colab_part1_seed42',
    'outputs/part1_btc_macro_state_outputs/part1_btc_macro_state/colab_part1_seed42',
)
PART2_RUN_DIR = choose_existing(
    'outputs/part2_portfolio_risk_budget/colab_part2_seed42',
    'outputs/part2_portfolio_risk_budget_outputs/part2_portfolio_risk_budget/colab_part2_seed42',
)
OUTPUT_DIR = Path('outputs/part11_hmm_model_comparison_outputs/part11_hmm_model_comparison')
RUN_ID = 'colab_part11_seed42'

print('INPUT_DIR:', INPUT_DIR)
print('PART1_RUN_DIR:', PART1_RUN_DIR)
print('PART2_RUN_DIR:', PART2_RUN_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
assert PART1_RUN_DIR.exists(), 'Part 1 run directory not found. Run Part 1 first.'
assert PART2_RUN_DIR.exists(), 'Part 2 run directory not found. Run Part 2 first.'

In [ ]:
!python experiments/part11_hmm_model_comparison/run_part11_hmm_model_comparison.py \
  --input-dir "{INPUT_DIR}" \
  --part1-run-dir "{PART1_RUN_DIR}" \
  --part2-run-dir "{PART2_RUN_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --run-id "{RUN_ID}" \
  --seed 42

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
display(pd.read_csv(RESULTS / 'part11_model_selection_summary.csv'))
display(pd.read_csv(RESULTS / 'part11_state_profile_summary.csv'))
display(pd.read_csv(RESULTS / 'part11_state_mapping_stability.csv'))
display(pd.read_csv(RESULTS / 'part11_rule_performance_by_model.csv'))

In [ ]:
findings = json.loads((RESULTS / 'part11_model_comparison_key_findings.json').read_text())
print(json.dumps(findings, indent=2))

In [ ]:
from IPython.display import Image, display

for name in [
    'part11_aic_bic_by_model.png',
    'part11_min_state_share_by_model.png',
    'part11_hmm4_vs_hmm5_state_timeline.png',
    'part11_rule_result_by_model.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)